# Version 2: longitudinal player–champion collection

**Implementation only — no cells have been executed.** Network collection, offline feature processing, and output exports are disabled below. Imports never contact Riot. This notebook uses the existing `RIOT_API_KEY` / `X-Riot-Token` convention, and never prints credentials.

Install `pandas`, `requests`, `python-dotenv`, and `pyarrow` in your environment separately if needed. No installation cell is included. Open this notebook with the repository root or `hlr_model/` as the working directory.

The study is NA / Ranked Solo-Duo / patch **16.19**. V1's parser takes the first two components of `gameVersion`; inspected cached values include `16.19.820.7193`, so no alternative year-to-patch conversion is used. No timelines, OP.GG, LoLalytics collection, matchup merge, or model fitting is part of this notebook.


## 1. Imports and configuration

The finalized plan is **200 NA Master-labeled focal players, queue 420, patch 16.19, and up to 170 preceding completed games per player from the 2026 ranked season**. The January 8 boundary limits how far back history may search; it does not instruct the pipeline to download the entire season. No player or champion minimum-game eligibility threshold is applied before EDA.

Target discovery uses V1's pre-deployment lower bound, **2026-09-23 12:00 UTC**, and a finite upper bound through **October 7, 2026 in NA**. Riot schedules the next patch on October 7; the API upper bound is the following midnight in Pacific time so we do not guess the deployment hour. Actual `gameVersion` remains the patch-membership criterion. History queries start at **January 8, 2026, noon PST**, matching Riot's local-server ranked-season start. These are seconds for Match-V5 requests; participant-record timestamps and overlap checks use milliseconds.

Sources: [Riot's 2026 patch schedule](https://support.riotgames.com/en-us/league-of-legends/gameplay/patch-schedule-league-of-legends) and [ranked-season start in patch 26.1 notes](https://www.leagueoflegends.com/en-us/news/game-updates/patch-26-1-notes/). Public patch names differ from the cached Match-V5 major/minor identifier; the verified target remains `16.19`.

Change count, depth, seed, boundaries, and output folder here. Keep `RUN_COLLECTION`, `RUN_OFFLINE_PROCESSING`, and `SAVE_OUTPUTS` disabled while reviewing. The initial history expands with completed target games, without dropping older games.

`ALLOW_UNVERIFIED_MASTERS=True` permits explicitly flagged V1 seeds whose study-period membership lacks dated evidence. Set it to `False` to require documented membership. Optional rank evidence CSV columns are `puuid,tier,patch,observed_at_utc,evidence_source`. Today's league membership is not historical evidence.

The run freezes its end at the earlier of collection initialization and the patch search upper bound. Resume with unchanged configuration and output folder; an earlier incompatible checkpoint requires a new run folder. Existing datasets and checkpoints are not overwritten to migrate configurations.


In [ ]:
from pathlib import Path
import os
from datetime import datetime
import sys
import pandas as pd
from dotenv import load_dotenv

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [start, *start.parents]
                     if (p / "lpm_model" / "otp_match_pipeline.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hlr_model.riot_api_helpers import RiotClient, read_json
from hlr_model.longitudinal_pipeline import (
    Config, CollectionRun, candidate_players, select_cohort,
    expanding_features, quality_report, save_outputs,
)

RUN_COLLECTION = False
RUN_OFFLINE_PROCESSING = False
SAVE_OUTPUTS = False
EXPORT_CSV = False
PLAYER_COUNT = 200
HISTORICAL_DEPTH = 170
RANDOM_SEED = 1619
ALLOW_UNVERIFIED_MASTERS = True
# Jan 8 at noon PST: Riot's ranked-season start in NA local server time.
SEASON_START = int(datetime.fromisoformat("2026-01-08T12:00:00-08:00").timestamp())
# Existing V1 pre-patch lower bound (2026-09-23 12:00 UTC).
TARGET_SEARCH_START = 1790164800
# Include all of Oct 7 in NA: patch rollout hour is not assumed.
TARGET_SEARCH_END = int(datetime.fromisoformat("2026-10-08T00:00:00-07:00").timestamp())
OTP_SHARE_THRESHOLD = 0.50
OUTPUT_RUN = PROJECT_ROOT / "hlr_model" / "data" / "runs" / "patch_16_19_seed_1619"
HISTORICAL_RANK_EVIDENCE = None  # Or a Path to a documented historical rank CSV.

config = Config(player_count=PLAYER_COUNT, historical_depth=HISTORICAL_DEPTH,
                random_seed=RANDOM_SEED, allow_unverified_masters=ALLOW_UNVERIFIED_MASTERS,
                season_start=SEASON_START, target_search_start=TARGET_SEARCH_START,
                target_search_end=TARGET_SEARCH_END,
                otp_share_threshold=OTP_SHARE_THRESHOLD)
PROCESS_DATA = RUN_COLLECTION or RUN_OFFLINE_PROCESSING


## 2. Paths, environment variables, and cache initialization

The root `.env` is read explicitly. Existing caches are referenced in place: `data/otp_raw_matches/` and, if present, `lpm_model/data/otp_raw_matches/`. V1 files are never written. New match payloads and immutable-query ID pages live under this run's `raw/`, and the cohort, frozen query cutoff, completion statuses, and failures live in `metadata/checkpoint.json`.

Use one collector per output folder and API key. The sequential client applies conservative 20/second and 100/120-second limits, honors advertised app/method limits and counts, handles `Retry-After`, retries transient failures with bounded backoff, and stops immediately on 401/403. Actual key limits may differ; response headers and 429 take precedence. Importing V1 is safe: only its pure `patch_from_match` and `ROLE_MAP` are reused.


In [ ]:
load_dotenv(PROJECT_ROOT / ".env", override=False)
client = RiotClient(os.getenv("RIOT_API_KEY"), enabled=RUN_COLLECTION)
checkpoint_path = OUTPUT_RUN / "metadata" / "checkpoint.json"
print("Output run:", OUTPUT_RUN.relative_to(PROJECT_ROOT))
print("Network collection enabled:", RUN_COLLECTION)
# No directories or caches are created by this cell.


## 3. Selecting 200 Masters players

Existing `seed_players.csv` and labeled V1 indexes supply candidates. The inspected seed table has 3,000 rows: 2,000 Master, 700 Grandmaster, 300 Challenger. It contains **no rank observation date**. Neither legacy indexes nor match participation prove patch-period rank. Missing tier labels are never converted to Master, unlike V1's migration fallback.

Documented historical evidence is preferred. Within verified and unverified pools, sorting PUUIDs followed by seeded random sampling makes selection reproducible and independent of win rate, OTP status, match availability, or opponent rank. If fewer than 200 eligible candidates exist, collection stops instead of silently reducing the cohort or replacing it with today's Masters. An existing run restores its exact saved cohort.


In [ ]:
if checkpoint_path.exists():
    saved = read_json(checkpoint_path)
    cohort = pd.DataFrame(saved["identity"]["cohort"])
else:
    candidates = candidate_players(PROJECT_ROOT, config, HISTORICAL_RANK_EVIDENCE)
    cohort = select_cohort(candidates, config)

run = CollectionRun(PROJECT_ROOT, OUTPUT_RUN, config, cohort,
                    client=client, enabled=RUN_COLLECTION)
if RUN_COLLECTION:
    run.save()  # Persist cohort and frozen end time before the first HTTP request.
print("Selected players:", len(cohort))
print("Historically verified Masters:", int(cohort.historical_masters_verified.sum()))
if not cohort.historical_masters_verified.all():
    print("LIMITATION: some V1 Master labels lack documented patch-period membership.")
display(cohort[["selection_source", "historical_masters_verified", "rank_limitation"]].value_counts().to_frame("players"))


## 4. Retrieving patch 16.19 match IDs

Paginate queue-420 IDs **only within the configured patch search window**, using a frozen `endTime`. There is no target match-count cap. Merge V1's patch IDs and recover focal-player appearances from existing patch JSONs, then validate metadata and timestamps against the same boundaries. This stage does not scan or download the season's non-target matches.

Reading/downloading candidate details is necessary here: IDs alone cannot identify `gameVersion` or the earliest target timestamp needed in section 5. Shared JSONs are downloaded once globally. Wrong-patch/queue and out-of-bound records are excluded and counted; a missing target detail or incomplete focal participant keeps discovery incomplete so an uncertain anchor is not used. Successful players are skipped on resume; failed players reuse cached pages and payloads.


In [ ]:
if RUN_COLLECTION:
    run.collect_targets()
else:
    print("Target collection disabled; no Riot requests.")


## 5. Retrieving 170 preceding historical match IDs

Anchor history at the earliest valid patch-16.19 target, including targets with missing lane opponents. Request newest-first Ranked Solo/Duo IDs between the **January 8 season start** and the anchor's second. Include that second in the coarse API upper bound, then require exact millisecond `timestamp < anchor` and **`end_timestamp < anchor`**. Games overlapping the first target cannot supply its features.

Request at most the remaining number of eligible games: normally 100 IDs, then 70. Process IDs in Riot's newest-first order, sort retained records chronologically, and stop immediately at 170 valid completed games. Ineligible overlapping games may require a few extra IDs. Stop earlier if the season boundary is encountered or Riot returns a short/empty page within the bounded query. No older pages are requested after the 170-game requirement is met, and no history from before the season start is retained.

Preserve histories shorter than 170 and every selected player without applying minimum-game filters. `history_stop_reason`, `genuinely_short_history`, `history_incomplete`, discovery completeness, exclusions, and missing-payload counts distinguish bounded API exhaustion from errors. A download failure preserves the available partial history, marks it incomplete, and does not substitute an older game; resumption retries the same cached query.


In [ ]:
if RUN_COLLECTION:
    run.collect_histories()
else:
    print("Historical collection disabled; no Riot requests.")


## 6. Deduplicating and downloading missing matches

The earlier stages already needed metadata to establish patch and cutoff eligibility. This pass ensures the final union of history/target IDs is present. A JSON contains every participant and is fetched once globally, including matches shared by multiple focal players. New files use atomic writes; corrupt legacy files remain untouched. Interrupted writes are not treated as complete matches. Failed requests are recorded and can be retried by rerunning sections 4–6.


In [ ]:
if RUN_COLLECTION:
    print("Unique required matches:", run.download_required())
else:
    print("Detail collection disabled; cached JSONs are available for opt-in offline processing.")


## 7. Building historical player-match records

Build separate history and target tables with one row per focal player/match. Preserve separate player rows when multiple selected players share a match. Unknown or duplicate `teamPosition` assignments produce a missing lane matchup, never a guessed opponent. No assumption is made about opponents' rank.

Riot's `teamPosition` is inferred from actual play; it is useful for matchup indexing but is not verified pregame lane intent. Champion, team side, summoner spells, and rune choices are retained. Rune `var1/var2/var3` postgame counters are explicitly stripped. Match start/end timestamps are audit fields. Win is the response; KDA, damage, gold, and other postgame performance variables are not included as predictors. Keep timestamps and missingness/collection flags out of model predictors unless separately justified.


In [ ]:
if PROCESS_DATA:
    historical_matches, target_matches, player_coverage = run.build_records()
    display(player_coverage.describe(include="all"))
else:
    print("Feature processing disabled. Enable RUN_OFFLINE_PROCESSING for a cached run.")


## 8. Constructing expanding-window features

The first target uses only the selected initial history. Each subsequent target uses that history plus all earlier target games whose outcomes were available strictly before kickoff. Matches with equal timestamps cannot contribute to each other's features. Counts never drop as the window expands.

Retain total, champion, main-champion, and other-champion game/win counts, raw win rates, usage shares, and the raw champion-minus-overall win-rate difference. Separate `*_wr_beta11` columns use the simple Beta(1,1) estimate `(wins+1)/(games+2)`; raw rates stay missing for zero games. No fitted proficiency score is introduced.

The prior main is the most-used champion; ties use the smallest champion ID. `is_otp` means **this observation plays the prior main**, with prior usage share at least `OTP_SHARE_THRESHOLD` and **no minimum-game threshold**. This is a configurable operational definition, not a claim of a universal OTP cutoff. It never uses future history.


In [ ]:
if PROCESS_DATA:
    longitudinal_features = expanding_features(historical_matches, target_matches, config)
    display(longitudinal_features.head())


## 9. Building the final longitudinal dataset

Attach cohort selection and collection completeness metadata without multiplying player-match rows. Short and incomplete histories are retained and separately flagged for review; they are not silently presented as a complete 170-game baseline. Do not fit a model until you review these flags. No LoLalytics data is merged.


In [ ]:
if PROCESS_DATA:
    cohort_output = cohort.merge(player_coverage, on="puuid", how="left", validate="one_to_one")
    final_longitudinal = longitudinal_features.merge(
        cohort_output, on="puuid", how="left", validate="many_to_one")


## 10. Data quality checks and summary statistics

On your eventual run, check player/match uniqueness, target patch and queue, patch search bounds, historical season cutoff, a maximum of 170 initial games per player, historical/target separation, strict completion-before-kickoff, nondecreasing expanding counts, valid win/game counts, and preservation of all target rows. Report genuinely short histories separately from incomplete histories due to errors, as well as missing matchups, exclusions, per-player counts, failures, and cache/request counters. Codex has not executed these checks or any tests.

“All available” refers to API-returned and reusable cached patch matches in the configured study window. Riot retention, unavailable payloads, late indexing, and collection failures can still limit completeness. Exhausted bounded pagination confirms no more returned games inside the 2026-season query; it does not establish lifetime history or diagnose whether Riot retained every played game. Rank evidence and inferred `teamPosition` retain their existing limitations.


In [ ]:
if PROCESS_DATA:
    summary = quality_report(cohort, historical_matches, target_matches,
                             final_longitudinal, player_coverage, config)
    display(pd.Series(summary, name="value"))
    display(player_coverage)
    print("Cache counters this session:", dict(run.store.metrics))
    print("HTTP counters this session:", dict(client.metrics))
    print("Recorded failure attempts:", len(run.state["failures"]))


## 11. Saving output datasets

Set `SAVE_OUTPUTS=True` when you want exports. Every save creates a new timestamped directory under `OUTPUT_RUN/processed/`, so previous datasets are never overwritten. Parquet outputs: `player_cohort`, `historical_matches`, `target_matches`, `longitudinal_modeling`, and `player_coverage`. Optional CSVs use the same stems. `collection_metadata.json` records configuration, frozen end time, quality summary, failures, cache metrics, and API metrics. Counters describe this notebook session; ID/checkpoint artifacts persist across sessions.

**Later execution:** review section 1, then set `RUN_COLLECTION=True` and run sections 1–10 in order. Set `SAVE_OUTPUTS=True` to run section 11. To resume, keep the same configuration and `OUTPUT_RUN`, restart the kernel, and rerun in order. To rebuild from cached results without HTTP, use `RUN_COLLECTION=False`, `RUN_OFFLINE_PROCESSING=True`; use `SAVE_OUTPUTS=True` only to export. Interrupt collection with the notebook's kernel interrupt, then resume from the same folder. Keep V1 files and caches in place.

**Request/runtime planning:** 200 players × 170 historical games yields at most **34,000 historical player-match rows**, with approximately **400 history-ID requests** (100 + 70 per player) if every returned game is eligible. Short histories reduce this; overlaps or other exclusions can add a few ID/detail requests. Match sharing and valid cached JSONs reduce new detail downloads.

If `T` is the average target count, allow approximately `200 × (floor(T/100) + 1)` target-ID requests, plus details for the unique uncached targets and a few wrong-patch candidates near rollout boundaries. There is no entire-season discovery/download term. With no cache reuse or match sharing and `T=20`, a planning estimate is **34,000 history details + 400 history-ID requests + 4,000 target details + 200 target-ID requests = 38,600 requests**, plus any exclusions/retries. At 100 requests/120 seconds that is about **12.9 hours** of rate-limit budget before latency. No API requests or data collection have been started.


In [ ]:
if SAVE_OUTPUTS:
    if not PROCESS_DATA:
        raise RuntimeError("Enable collection or offline processing and run sections 7–10 before saving.")
    output_path = save_outputs(run, cohort_output, historical_matches, target_matches,
                              final_longitudinal, player_coverage, summary,
                              enabled=SAVE_OUTPUTS, csv=EXPORT_CSV)
    print("Saved output snapshot:", output_path.relative_to(PROJECT_ROOT))
else:
    print("Saving disabled; no output datasets written.")
